# DMRG算法教程

深入理解密度矩阵重整化群(DMRG)算法。

## 学习目标

1. 理解DMRG算法原理
2. 实现简单的DMRG扫描
3. 理解与MPS的关系
4. 分析收敛性

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../../common')

%matplotlib inline

## 1. DMRG算法概述

DMRG是计算一维量子多体系统基态的强大算法。

### 核心思想
1. 将系统分为左右两块
2. 在两块界面处对角化有效哈密顿量
3. 保留最重要的态（最大施密特值）
4. 扫描整个系统直到收敛

### 与MPS的关系
DMRG = MPS + 变分优化

In [ ]:
# DMRG示意图
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 6))

# 系统分块
L = 10
center = L // 2

# 左块
for i in range(center):
    rect = FancyBboxPatch((i, 0), 0.8, 0.8, 
                          boxstyle="round,pad=0.05",
                          facecolor='lightblue', 
                          edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(i + 0.4, 0.4, f'{i+1}', ha='center', va='center', fontsize=10)

# 右块
for i in range(center, L):
    rect = FancyBboxPatch((i, 0), 0.8, 0.8, 
                          boxstyle="round,pad=0.05",
                          facecolor='lightcoral', 
                          edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(i + 0.4, 0.4, f'{i+1}', ha='center', va='center', fontsize=10)

# 中心位置标记
arrow = FancyArrowPatch((center + 0.4, -0.5), (center + 0.4, -0.1),
                       arrowstyle='->', mutation_scale=20, 
                       color='red', linewidth=3)
ax.add_patch(arrow)
ax.text(center + 0.4, -0.8, 'DMRG Center', ha='center', 
        fontsize=12, fontweight='bold', color='red')

# 标签
ax.text(center/2, 1.2, 'Left Block', ha='center', fontsize=14, fontweight='bold')
ax.text(center + (L-center)/2, 1.2, 'Right Block', ha='center', fontsize=14, fontweight='bold')

ax.set_xlim(-0.5, L)
ax.set_ylim(-1.5, 1.8)
ax.axis('off')
ax.set_title('DMRG System Partition', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. DMRG扫描过程

In [ ]:
# 模拟DMRG能量收敛
def simulate_dmrg_convergence(n_sweeps=10, L=20):
    """模拟DMRG收敛过程"""
    # 精确能量（理论值）
    E_exact = -L * 0.5
    
    energies = []
    errors = []
    
    for sweep in range(n_sweeps):
        # 模拟指数收敛
        E_current = E_exact + (E_exact * 0.1) * np.exp(-sweep * 0.8)
        energies.append(E_current)
        errors.append(abs(E_current - E_exact))
    
    return energies, errors, E_exact

energies, errors, E_exact = simulate_dmrg_convergence(n_sweeps=15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 能量收敛
ax1.plot(energies, 'o-', markersize=8, linewidth=2, label='DMRG Energy')
ax1.axhline(E_exact, color='red', linestyle='--', linewidth=2, label='Exact')
ax1.set_xlabel('Sweep Number', fontsize=12)
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title('DMRG Energy Convergence', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 误差（对数）
ax2.semilogy(errors, 'o-', markersize=8, linewidth=2, color='orange')
ax2.set_xlabel('Sweep Number', fontsize=12)
ax2.set_ylabel('Energy Error (log)', fontsize=12)
ax2.set_title('Convergence Error', fontsize=13)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("DMRG收敛分析:")
print(f"初始误差: {errors[0]:.2e}")
print(f"最终误差: {errors[-1]:.2e}")
print(f"收敛因子: {errors[-1]/errors[0]:.2e}")

## 3. 键维度与精度的关系

In [ ]:
# 研究chi与误差的关系
chi_values = np.array([4, 8, 16, 32, 64, 128, 256])

# 临界点需要更大chi
errors_ordered = 1e-2 * np.exp(-chi_values / 10)
errors_critical = 1e-1 * np.exp(-chi_values / 30)

plt.figure(figsize=(10, 6))
plt.loglog(chi_values, errors_ordered, 'o-', markersize=8, 
          linewidth=2, label='Ordered Phase')
plt.loglog(chi_values, errors_critical, 's-', markersize=8, 
          linewidth=2, label='Critical Point')
plt.axhline(1e-10, color='red', linestyle='--', 
           label='Target Precision', alpha=0.5)

plt.xlabel('Bond Dimension $\chi$', fontsize=12)
plt.ylabel('Truncation Error', fontsize=12)
plt.title('Bond Dimension vs Accuracy', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

print("键维度选择建议:")
print("有序相: χ = 16-32 通常足够")
print("临界点: χ = 64-128 或更大")
print("2D系统: χ = 100-1000")

## 4. DMRG vs 精确对角化

In [ ]:
# 计算复杂度比较
L_values = np.arange(5, 51, 5)
chi = 64

# 精确对角化: O(2^3L)
complexity_ED = 2**(3 * L_values)

# DMRG: O(L * chi^3)
complexity_DMRG = L_values * chi**3

plt.figure(figsize=(10, 6))
plt.semilogy(L_values, complexity_ED, 'o-', markersize=8, 
            linewidth=2, label='Exact Diagonalization')
plt.semilogy(L_values, complexity_DMRG, 's-', markersize=8, 
            linewidth=2, label=f'DMRG (χ={chi})')

plt.xlabel('System Size $L$', fontsize=12)
plt.ylabel('Computational Cost (arbitrary units)', fontsize=12)
plt.title('DMRG vs Exact Diagonalization Complexity', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

print("复杂度分析:")
print(f"L=20时:")
print(f"  ED: ~{2**(3*20):.2e} 操作")
print(f"  DMRG: ~{20 * chi**3:.2e} 操作")
print(f"  加速比: ~{2**(3*20) / (20 * chi**3):.2e}x")

## 练习

1. 实现单site DMRG算法
2. 比较finite vs infinite DMRG
3. 研究不同初始态对收敛的影响
4. 实现时间演化DMRG (t-DMRG)

## 参考文献

- White (1992) - DMRG原始论文
- Schollwöck (2011) - 现代DMRG综述
- Verstraete & Cirac (2004) - DMRG与MPS